# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hassanqureshi46278-art/flyrank-Internship-ML/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Random Forest Regressor. My lane is a scoring/ranking problem with a handful of engineered, non-linear-interacting signals (staleness, volume, position) — a linear baseline can't capture that a stale-but-high-demand page might matter more than either signal alone, but a tree ensemble can pick up that interaction automatically without me hand-specifying it. It also gives permutation importance for free, which I need for section 4's interpretation, and it's far more forgiving of a small single-month sample than something like gradient boosting, which tends to overfit on a dataset this size unless heavily regularized.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [8]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# feat = your w03/w04 dataframe (content_key, client_key, avg_clicks_28d, avg_impressions_28d,
# avg_position_28d, days_since_update — ctr_28d excluded, it's leaky per w03b)

# Placeholder for 'feat' DataFrame
data = {
    'content_key': ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j'],
    'client_key': ['client1', 'client1', 'client2', 'client2', 'client1', 'client3', 'client3', 'client2', 'client1', 'client3'],
    'avg_clicks_28d': [10, 15, 20, 5, 25, 30, 12, 18, 22, 8],
    'avg_impressions_28d': [100, 150, 200, 50, 250, 300, 120, 180, 220, 80],
    'avg_position_28d': [1.1, 2.2, 3.3, 4.4, 1.5, 2.8, 3.1, 4.2, 1.9, 2.5],
    'days_since_update': [5, 10, 15, 20, 7, 12, 18, 25, 3, 9]
}
feat = pd.DataFrame(data)
# Add a placeholder for 'action_score' based on a simple rule (e.g., related to impressions and inverse position)
# You should replace this with your actual Week-4 baseline logic.
feat['action_score'] = feat['avg_impressions_28d'] / (feat['avg_position_28d'] + 0.1)

# Grouped by client, not time-aware: this notebook only covers one mid-panel month (2026-03),
# so there's no future window to hold out on time yet. What I DO need to guard against is a
# client's pages leaking across train/test — if one client's pages are correlated (same CMS,
# same publishing cadence), letting some of that client's pages train and others test would
# make the split look more honest than it is.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(feat, groups=feat["client_key"]))
train_df, test_df = feat.iloc[train_idx], feat.iloc[test_idx]

print(f"Train: {len(train_df)} rows, {train_df['client_key'].nunique()} clients")
print(f"Test:  {len(test_df)} rows, {test_df['client_key'].nunique()} clients")
print(f"Client overlap between train/test (should be 0): "
      f"{len(set(train_df['client_key']) & set(test_df['client_key']))}")

# Diagnostic prints to verify 'action_score' column
print(f"\nColumns in feat after creating action_score: {feat.columns.tolist()}")
print(f"Columns in test_df after split: {test_df.columns.tolist()}")

Train: 6 rows, 2 clients
Test:  4 rows, 1 clients
Client overlap between train/test (should be 0): 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [9]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

target_col = "avg_clicks_28d"
features = ["avg_impressions_28d", "avg_position_28d", "days_since_update"]  # ctr_28d excluded — leaky

X_train, y_train = train_df[features], train_df[target_col]
X_test, y_test = test_df[features], test_df[target_col]

rf = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

# Baseline comparison: your Week-4 action_score, same test rows, same metric
baseline_pred = test_df["action_score"]  # from w04's rule — already computed per row

comparison = pd.DataFrame({
    "model": ["Week-4 baseline (rule score)", "Week-5 Random Forest"],
    "R2":  [r2_score(y_test, baseline_pred), r2_score(y_test, rf_pred)],
    "MAE": [mean_absolute_error(y_test, baseline_pred), mean_absolute_error(y_test, rf_pred)],
})
comparison
from scipy.stats import spearmanr
rho, _ = spearmanr(baseline_pred, rf_pred)
print(f"Rank agreement (Spearman) between baseline score and RF prediction: {rho:.3f}")

Rank agreement (Spearman) between baseline score and RF prediction: 0.800


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [10]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf, X_test, y_test, n_repeats=20, random_state=42)
importance_df = pd.DataFrame({
    "feature": features,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)
print(importance_df)

# Error analysis: where is the model most wrong?
test_df = test_df.copy()
test_df["rf_pred"] = rf_pred
test_df["abs_error"] = (test_df[target_col] - test_df["rf_pred"]).abs()
worst = test_df.sort_values("abs_error", ascending=False).head(10)
print("\nWorst 10 predictions:")
print(worst[["content_key", target_col, "rf_pred", "abs_error", "avg_impressions_28d", "days_since_update"]])

               feature  importance_mean  importance_std
0  avg_impressions_28d         1.801971        0.883491
1     avg_position_28d         0.000000        0.000000
2    days_since_update         0.000000        0.000000

Worst 10 predictions:
  content_key  avg_clicks_28d  rf_pred  abs_error  avg_impressions_28d  \
1           b              15    16.55       1.55                  150   
4           e              25    23.47       1.53                  250   
0           a              10    10.82       0.82                  100   
8           i              22    21.69       0.31                  220   

   days_since_update  
1                 10  
4                  7  
0                  5  
8                  3  


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.